In [13]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
# 1. 데이터 세트 불러오기 (diabetes_data.csv)
url = 'https://drive.google.com/uc?export=download&id=1zs-H7slk3aAkGNsg7TFxK2sS7OjHxeku'

df = pd.read_csv(url)

print(df.shape)
df.head()

plt.figure(figsize=(15,12))
# sns.heatmap(df.corr(),linewith =0.1 ,vm

(520, 16)


<Figure size 1500x1200 with 0 Axes>

<Figure size 1500x1200 with 0 Axes>

In [17]:
# ============================================================
# 필요한 라이브러리 불러오기
# ============================================================

# Iris(붓꽃) 데이터셋 불러오기
from sklearn.datasets import load_iris

# 데이터를 학습용(train) / 테스트용(test)으로 나누기
from sklearn.model_selection import train_test_split

# 데이터를 0~1 범위로 정규화
from sklearn.preprocessing import MinMaxScaler

# ★ 추가
# 평균 0, 표준편차 1을 기준으로 표준화
from sklearn.preprocessing import StandardScaler

# 로지스틱 회귀 모델
from sklearn.linear_model import LogisticRegression


# ============================================================
# 기본 설정
# ============================================================

# 랜덤 시작값(seed)
# train/test 데이터를 랜덤하게 나눌 때
# 실행할 때마다 동일한 데이터가 선택되도록 고정
# 학습데이터가 바뀜
random_state = 1234

# 로지스틱 회귀 학습 시 최대 반복 횟수
# 모델이 최적의 값을 찾을 때 최대 5000번까지 반복
# 무조건 5000번 반복한다는 뜻은 아님
iter_max = 5000


# ============================================================
# Iris 데이터 불러오기
# ============================================================

# as_frame=True
# → 데이터를 pandas DataFrame 형태로 가져옴
iris = load_iris(as_frame=True)

# Xi = 입력 데이터(문제)
# 꽃받침 길이, 꽃받침 너비,
# 꽃잎 길이, 꽃잎 너비
#
# Yi = 정답 데이터
# 0 = setosa
# 1 = versicolor
# 2 = virginica
Xi, Yi = iris.data, iris.target


# ============================================================
# ★ 가장 좋은 결과 저장
# ============================================================

best_score = 0
best_stat = 0
best_c = 0
best_scaler = ""
best_solver = ""


# ============================================================
# test_size를 변경하면서 성능 비교
# ============================================================

# test_size
#
# 0.1 → train 90%, test 10%
# 0.3 → train 70%, test 30%
# 0.5 → train 50%, test 50%
# 0.7 → train 30%, test 70%
#
# 학습 데이터와 테스트 데이터의 비율에 따라
# 정확도가 어떻게 변하는지 확인
for stat in [0.1, 0.3, 0.5, 0.7]:

    # --------------------------------------------------------
    # 1. Train / Test 데이터 분리
    # --------------------------------------------------------

    xi_train, xi_test, yi_train, yi_test = train_test_split(
        Xi,
        Yi,

        # 테스트 데이터로 사용할 비율
        test_size=stat,

        # Iris의 클래스 0, 1, 2 비율을
        # train과 test에서도 비슷하게 유지
        stratify=Yi,

        # 랜덤 결과 고정
        # 같은 값이면 실행할 때마다 같은 방식으로 데이터 분리
        random_state=random_state
    )


    # ========================================================
    # ★ MinMaxScaler / StandardScaler 둘 다 비교
    # ========================================================

    for scaler_name, scaler in [
        ("MinMax", MinMaxScaler()),
        ("Standard", StandardScaler())
    ]:

        # --------------------------------------------------------
        # 2. 데이터 정규화
        # --------------------------------------------------------

        # Min-Max 정규화 객체 생성
        # 각 특성의 값을 일반적으로 0~1 범위로 변환
        #
        # ★ StandardScaler도 같이 비교
        # StandardScaler는 평균 0, 표준편차 1을 기준으로 변환


        # fit
        # → train 데이터의 최소값/최대값을 학습
        #
        # transform
        # → 학습한 최소값/최대값으로 데이터를 0~1로 변환
        #
        # fit_transform
        # → fit + transform을 한 번에 수행
        xi_train_scaled = scaler.fit_transform(xi_train)


        # test 데이터에서는 fit을 다시 하면 안 됨
        #
        # train에서 구한 최소값/최대값을 그대로 사용해서
        # test 데이터를 변환
        #
        # test 데이터까지 fit에 사용하면
        # 데이터 누수(Data Leakage)가 발생할 수 있음
        xi_test_scaled = scaler.transform(xi_test)


        # --------------------------------------------------------
        # 3. C값을 변경하면서 로지스틱 회귀 성능 비교
        # --------------------------------------------------------

        # C = 규제(Regularization)의 정도를 조절하는 값
        #
        # C가 작음
        # → 규제가 강함
        # → 모델을 단순하게 만듦
        # → 너무 작으면 과소적합(Underfitting) 가능
        #
        # C가 큼
        # → 규제가 약함
        # → 학습 데이터에 더 적극적으로 맞춤
        # → 너무 크면 과적합(Overfitting) 가능
        #
        # 여러 C값을 사용해서 어떤 값이 좋은지 비교

        # ★ 기존보다 C를 더 세밀하게 비교
        for c in [
            0.01, 0.05, 0.1, 0.5,
            1, 2, 5, 10, 20, 50,
            100, 500, 1000
        ]:


            # ====================================================
            # ★ 학습 알고리즘(solver)도 변경하면서 비교
            # ====================================================

            for solver in ["lbfgs", "newton-cg", "saga"]:

                # ----------------------------------------------------
                # 4. 로지스틱 회귀 모델 생성
                # ----------------------------------------------------

                mi = LogisticRegression(

                    # 규제 정도 조절
                    C=c,

                    # ★ 최적의 계수를 찾는 알고리즘!!
                    solver=solver,

                    # 학습 최대 반복 횟수
                    max_iter=iter_max,

                    # ★ saga에서 랜덤 결과 고정
                    random_state=random_state
                )


                # ----------------------------------------------------
                # 5. 모델 학습
                # ----------------------------------------------------

                # xi_train = 문제(꽃의 특징)
                # yi_train = 정답(꽃의 종류)
                #
                # 입력 데이터와 정답을 이용해
                # 로지스틱 회귀 모델을 학습
                mi.fit(xi_train_scaled, yi_train)


                # ----------------------------------------------------
                # 6. Train 정확도 계산
                # ----------------------------------------------------

                # 모델이 자신이 학습한 데이터를
                # 얼마나 잘 맞추는지 확인
                train_score = mi.score(
                    xi_train_scaled,
                    yi_train
                )


                # ----------------------------------------------------
                # 7. Test 정확도 계산
                # ----------------------------------------------------

                # 학습할 때 사용하지 않은 test 데이터를 이용해서
                # 새로운 데이터도 잘 분류하는지 확인
                test_score = mi.score(
                    xi_test_scaled,
                    yi_test
                )


                # ----------------------------------------------------
                # 8. 결과 출력
                # ----------------------------------------------------

                print(
                    "test_size:", stat,                 # 테스트 데이터 비율
                    "scaler:", scaler_name,             # 정규화 방법
                    "C:", c,                            # 현재 C값
                    "solver:", solver,                  # 학습 알고리즘
                    "train:", round(train_score, 4),    # 학습 정확도
                    "test:", round(test_score, 4)       # 테스트 정확도
                )


                # ====================================================
                # ★ 지금까지 가장 높은 test 정확도 저장
                # ====================================================

                if test_score > best_score:
                    best_score = test_score
                    best_stat = stat
                    best_c = c
                    best_scaler = scaler_name
                    best_solver = solver

    # test_size 하나의 실험이 끝날 때 줄바꿈
    print()


# ============================================================
# ★ 가장 성능이 좋았던 설정 출력
# ============================================================

print("======================================")
print("가장 높은 Test 정확도")
print("======================================")

print("test_size :", best_stat)
print("train_size:", 1 - best_stat)
print("Scaler    :", best_scaler)
print("C         :", best_c)
print("solver    :", best_solver)
print("정확도    :", round(best_score, 4))

test_size: 0.1 scaler: MinMax C: 0.01 solver: lbfgs train: 0.7852 test: 0.7333
test_size: 0.1 scaler: MinMax C: 0.01 solver: newton-cg train: 0.7852 test: 0.7333
test_size: 0.1 scaler: MinMax C: 0.01 solver: saga train: 0.7852 test: 0.7333
test_size: 0.1 scaler: MinMax C: 0.05 solver: lbfgs train: 0.8519 test: 0.7333
test_size: 0.1 scaler: MinMax C: 0.05 solver: newton-cg train: 0.8519 test: 0.7333
test_size: 0.1 scaler: MinMax C: 0.05 solver: saga train: 0.8519 test: 0.6667
test_size: 0.1 scaler: MinMax C: 0.1 solver: lbfgs train: 0.8741 test: 0.7333
test_size: 0.1 scaler: MinMax C: 0.1 solver: newton-cg train: 0.8741 test: 0.7333
test_size: 0.1 scaler: MinMax C: 0.1 solver: saga train: 0.8741 test: 0.7333
test_size: 0.1 scaler: MinMax C: 0.5 solver: lbfgs train: 0.9259 test: 0.8667
test_size: 0.1 scaler: MinMax C: 0.5 solver: newton-cg train: 0.9259 test: 0.8667
test_size: 0.1 scaler: MinMax C: 0.5 solver: saga train: 0.9259 test: 0.8667
test_size: 0.1 scaler: MinMax C: 1 solver: lbf

In [ ]:
# 평균 -> 신뢰구간을 정하고 그 평균의 신뢰구간을 구한다 

In [18]:
# ============================================================
# 필요한 라이브러리
# ============================================================

# Iris 데이터
from sklearn.datasets import load_iris

# Train / Test 분리
from sklearn.model_selection import train_test_split

# 의사결정나무
from sklearn.tree import DecisionTreeClassifier


# ============================================================
# 기본 설정
# ============================================================

# 랜덤 결과 고정
random_state = 12345


# ============================================================
# Iris 데이터 불러오기
# ============================================================

iris = load_iris(as_frame=True)

# X = 입력 데이터
# Y = 정답
Xi, Yi = iris.data, iris.target


# ============================================================
# Test 비율을 변경하면서 성능 비교
# ============================================================

for stat in [0.1, 0.3, 0.5, 0.7]:

    # --------------------------------------------------------
    # 1. Train / Test 분리
    # --------------------------------------------------------

    xi_train, xi_test, yi_train, yi_test = train_test_split(
        Xi,
        Yi,

        # Test 데이터 비율
        test_size=stat,

        # 0, 1, 2 꽃 종류 비율 유지
        stratify=Yi,

        # 랜덤 결과 고정
        random_state=random_state
    )


    # --------------------------------------------------------
    # 2. 트리의 최대 깊이를 변경하면서 비교
    # --------------------------------------------------------

    # max_depth
    # → 의사결정나무가 최대 몇 층까지 내려갈 것인지
    #
    # 너무 작으면
    # → 모델이 너무 단순함
    # → 과소적합 가능
    #
    # 너무 크면
    # → 학습 데이터를 너무 자세하게 외움
    # → 과적합 가능

    for depth in [1, 2, 3, 4, 5, 10, None]:

        # ----------------------------------------------------
        # 3. 의사결정나무 생성
        # ----------------------------------------------------

        model = DecisionTreeClassifier(

            # 트리의 최대 깊이
            max_depth=depth,

            # 랜덤 결과 고정
            random_state=random_state
        )


        # ----------------------------------------------------
        # 4. 모델 학습
        # ----------------------------------------------------

        model.fit(
            xi_train,
            yi_train
        )


        # ----------------------------------------------------
        # 5. Train 정확도
        # ----------------------------------------------------

        train_score = model.score(
            xi_train,
            yi_train
        )


        # ----------------------------------------------------
        # 6. Test 정확도
        # ----------------------------------------------------

        test_score = model.score(
            xi_test,
            yi_test
        )


        # ----------------------------------------------------
        # 7. 결과 출력
        # ----------------------------------------------------

        print(
            "test_size:", stat,
            "depth:", depth,
            "train:", round(train_score, 4),
            "test:", round(test_score, 4)
        )

    print()

Task was destroyed but it is pending!
task: <Task pending name='Task-179' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/hangyeonghyeon/Desktop/Py/python_study/.venv/lib/python3.14/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-180' coro=<Kernel.shell_main() running at /Users/hangyeonghyeon/Desktop/Py/python_study/.venv/lib/python3.14/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/hangyeonghyeon/Desktop/Py/python_study/.venv/lib/python3.14/site-packages/zmq/eventloop/zmqstream.py:564]>
/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/inspect.py:2681: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  def __init__(self, name, kind, *, default=_empty, annotation=_empty):
Task was destroyed but it is pending!
task: <Task pending name='Task-180' coro=<Kernel.shell_main() running at /Users/hangyeong

test_size: 0.1 depth: 1 train: 0.6667 test: 0.6667
test_size: 0.1 depth: 2 train: 0.963 test: 0.9333
test_size: 0.1 depth: 3 train: 0.9852 test: 0.9333
test_size: 0.1 depth: 4 train: 0.9926 test: 0.9333
test_size: 0.1 depth: 5 train: 1.0 test: 0.9333
test_size: 0.1 depth: 10 train: 1.0 test: 0.9333
test_size: 0.1 depth: None train: 1.0 test: 0.9333

test_size: 0.3 depth: 1 train: 0.6667 test: 0.6667
test_size: 0.3 depth: 2 train: 0.9619 test: 0.9333
test_size: 0.3 depth: 3 train: 0.9714 test: 0.9556
test_size: 0.3 depth: 4 train: 0.9905 test: 0.9778
test_size: 0.3 depth: 5 train: 0.9905 test: 0.9778
test_size: 0.3 depth: 10 train: 1.0 test: 0.9778
test_size: 0.3 depth: None train: 1.0 test: 0.9778

test_size: 0.5 depth: 1 train: 0.6667 test: 0.6667
test_size: 0.5 depth: 2 train: 0.96 test: 0.9467
test_size: 0.5 depth: 3 train: 0.9733 test: 0.8933
test_size: 0.5 depth: 4 train: 1.0 test: 0.8933
test_size: 0.5 depth: 5 train: 1.0 test: 0.8933
test_size: 0.5 depth: 10 train: 1.0 test: 0.8